# Evaluation — Retrieval & Generation

A first real, numeric measurement of the RAG pipeline, instead of eyeballing individual answers. Two things are measured separately, because they can fail independently: whether **retrieval** finds the right source at all, and whether **generation** actually cites it correctly once retrieved.

**Honesty note on the eval set below:** these 6 questions are not client-provided — they are self-verified against our own corpus (`document_splits_v2.json`) by direct lookup this session, not trusted from any model-generated answer (a model-generated citation can look exactly right and still be wrong — see the Groq case study in `05_retrieval_langchain.ipynb`'s sibling test). This is a first measurement to build on, not a definitive benchmark. See "How to extend this" at the end.

In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb pandas


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import drive
import torch
drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())


## Eval set

92 real Arabic legal questions, each paired with the exact `(source, doc_id, article_no)` we confirmed by reading the real corpus JSON directly — not by trusting a generated answer's citation. Built in four passes:

1. **Questions 1–6** — labor, lease, and family law (the topics already tested elsewhere in this project).
2. **Questions 7–26** — deliberately span unrelated fields (aviation, military, health, antiquities, telecom, patents, and more) to check whether retrieval quality holds up outside familiar topics. This pass surfaced a real pattern: 5 misses, all on generic/procedural clauses rather than substantive rules.
3. **Questions 27–46** — built specifically to stress-test that pattern: effective-date lines, penalty clauses, quorum rules, incompatibility-of-office clauses, and a funds-to-treasury clause, pulled from 20 different laws that share very similar boilerplate wording. Each question still names its specific law, so this isn't an unfairly ambiguous bare clause — it's a fair test of whether retrieval can find the *right* law's version of a generic-sounding rule.
4. **Questions 47–92** — both categories doubled again (20 more substantive, 26 more boilerplate, from 46 different laws none of the first 46 questions touched) to check whether the 100% vs. ~35% split holds up at 2x the sample, or was partly a small-sample fluke.

Two candidate questions were dropped during construction because their `article_no` metadata didn't actually match the visible article number in the text — a real data-quality issue worth noting on its own, not just discarded silently.

In [ ]:
EVAL_SET = [
    {
        "question": "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "107"},
    },
    {
        "question": "متى يجوز للعامل انهاء عقد العمل دون اخطار؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "105"},
    },
    {
        "question": "هل يجب على الجهات الحكومية تزويد قاضي الدعوى العمالية بالمعلومات المطلوبة؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "128"},
    },
    {
        "question": "ما هي المدة التي يعتبر عقد الايجار منعقدا لها اذا لم يحدد الطرفان مدة العقد؟",
        "expected": {"source": "lloc", "doc_id": "K2714", "article_no": "4"},
    },
    {
        "question": "متى يحق للطفل الاختيار بين والديه في الحضانة؟",
        "expected": {"source": "lloc", "doc_id": "K1917", "article_no": "125"},
    },
    {
        "question": "متى تسقط حضانة الحاضن؟",
        "expected": {"source": "lloc", "doc_id": "K1917", "article_no": "136"},
    },
    # --- 20 additional questions, deliberately spanning fields far from labor/lease/family law,
    # to test whether retrieval quality holds up outside the topics tested so far. Same rule as
    # above: every expected citation was confirmed by reading the real corpus text directly.
    {  # public health / anti-smoking
        "question": "هل يجوز بيع منتجات التبغ لمن هم دون سن الثامنة عشرة؟",
        "expected": {"source": "lloc", "doc_id": "K0809", "article_no": "5"},
    },
    {  # utilities
        "question": "هل يحق للوزارة قطع خدمة الكهرباء عن المستهلك المتأخر عن السداد؟",
        "expected": {"source": "lloc", "doc_id": "L0196", "article_no": "6"},
    },
    {  # aviation
        "question": "من يتولى الاشراف والرقابة على شؤون الطيران المدني في البحرين؟",
        "expected": {"source": "lloc", "doc_id": "K1413", "article_no": "5"},
    },
    {  # military
        "question": "كم مهلة يمنح الضابط لتقديم دفاعه كتابة عند النظر في الاستغناء عن خدماته؟",
        "expected": {"source": "lloc", "doc_id": "L2000", "article_no": "21"},
    },
    {  # real estate registration
        "question": "هل يجوز نقل صحائف السجل العقاري خارج الجهاز المختص؟",
        "expected": {"source": "lloc", "doc_id": "K1313", "article_no": "12"},
    },
    {  # diplomatic corps
        "question": "على اي اساس تكون الترقية في وظائف السلك الدبلوماسي؟",
        "expected": {"source": "lloc", "doc_id": "K3709", "article_no": "16"},
    },
    {  # antiquities
        "question": "هل يجوز الكتابة او النقش على الاثار الثابتة؟",
        "expected": {"source": "lloc", "doc_id": "L1195", "article_no": "6"},
    },
    {  # hajj affairs
        "question": "ما الالتزام المفروض على المرخص له بتسيير حملة الحج تجاه الحجاج؟",
        "expected": {"source": "lloc", "doc_id": "L2676", "article_no": "7"},
    },
    {  # organ transplant / medical
        "question": "هل يجوز نقل عضو من جسم شخص حي اذا كان ذلك يؤدي الى وفاته؟",
        "expected": {"source": "lloc", "doc_id": "L1698", "article_no": "3"},
    },
    {  # public health / water
        "question": "ما هي الشروط الواجب توافرها في المياه داخل شبكة التوزيع؟",
        "expected": {"source": "lloc", "doc_id": "K3418", "article_no": "6"},
    },
    {  # elderly rights
        "question": "هل يجوز انشاء مؤسسة خاصة لرعاية المسنين دون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K5809", "article_no": "7"},
    },
    {  # engineering profession
        "question": "هل يجوز مزاولة المهنة الهندسية دون الحصول على ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K5114", "article_no": "2"},
    },
    {  # anti-doping convention
        "question": "هل يجوز لاي دولة طرف الانسحاب من الاتفاقية الدولية لمكافحة المنشطات؟",
        "expected": {"source": "lloc", "doc_id": "K1308", "article_no": "39"},
    },
    {  # GCC veterinary profession -- note: this law's article_no metadata uses Arabic-Indic digits
        "question": "هل يجوز ممارسة مهنة الطب البيطري دون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K1714", "article_no": "٢"},
    },
    {  # chamber of commerce
        "question": "اين يقع المقر الرئيسي لغرفة تجارة وصناعة البحرين؟",
        "expected": {"source": "lloc", "doc_id": "L4812", "article_no": "3"},
    },
    {  # traffic / vehicle registration
        "question": "هل تعتبر لوحات ارقام تسجيل المركبات ملكا خاصا لصاحب المركبة؟",
        "expected": {"source": "lloc", "doc_id": "K2314", "article_no": "12"},
    },
    {  # postal
        "question": "هل يجوز مراقبة المراسلات البريدية او الاطلاع عليها؟",
        "expected": {"source": "lloc", "doc_id": "K4914", "article_no": "6"},
    },
    {  # telecom
        "question": "لكم مدة يعين المدير العام لهيئة تنظيم الاتصالات؟",
        "expected": {"source": "lloc", "doc_id": "L4802", "article_no": "8"},
    },
    {  # vehicle accident compensation fund
        "question": "هل يغطي صندوق تعويض المتضررين من حوادث المركبات الاضرار التي تلحق بالممتلكات؟",
        "expected": {"source": "lloc", "doc_id": "K6114", "article_no": "6"},
    },
    {  # patents
        "question": "هل يجوز لطالب البراءة سحب طلبه قبل الاعلان عن قبوله؟",
        "expected": {"source": "lloc", "doc_id": "K0104", "article_no": "18"},
    },
    # --- 20 more questions, this time deliberately concentrated on the weak spot the first 26
    # exposed: generic, boilerplate-style clauses (effective-date lines, penalty clauses,
    # quorum rules, incompatibility-of-office clauses, funds-revert-to-treasury clauses) that
    # recur in near-identical wording across many unrelated laws. Each question names the
    # specific law so it isn't an unfairly ambiguous bare clause -- the test is whether
    # retrieval can still find the *right* law's version of a very generic-sounding rule.
    {  # effective date -- Chamber of Deputies internal bylaw
        "question": "متى يبدأ العمل بقانون اللائحة الداخلية لمجلس النواب؟",
        "expected": {"source": "lloc", "doc_id": "L5402", "article_no": "220"},
    },
    {  # effective date -- Shura Council internal bylaw
        "question": "متى يبدأ العمل بقانون اللائحة الداخلية لمجلس الشورى؟",
        "expected": {"source": "lloc", "doc_id": "L5502", "article_no": "191"},
    },
    {  # effective date -- GCC animal welfare law (article_no stored as Arabic-Indic digits)
        "question": "بعد كم يوما يصبح قانون الرفق بالحيوان لدول مجلس التعاون نافذا الزاميا؟",
        "expected": {"source": "lloc", "doc_id": "K5214", "article_no": "١٦"},
    },
    {  # effective date -- GCC veterinary preparations law
        "question": "متى يدخل قانون المستحضرات البيطرية لدول مجلس التعاون حيز النفاذ؟",
        "expected": {"source": "lloc", "doc_id": "K1914", "article_no": "38"},
    },
    {  # penalty clause -- children's restorative justice law
        "question": "ما عقوبة من يقدم معلومات كاذبة عن تعرض طفل لسوء المعاملة بموجب قانون العدالة الاصلاحية للاطفال؟",
        "expected": {"source": "lloc", "doc_id": "K0421", "article_no": "59"},
    },
    {  # penalty clause -- anti-commercial-fraud law
        "question": "ما عقوبة منع المفتشين من دخول المخازن للتفتيش بموجب قانون مكافحة الغش التجاري؟",
        "expected": {"source": "lloc", "doc_id": "K6214", "article_no": "13"},
    },
    {  # penalty clause -- consumer protection law
        "question": "ما عقوبة استيراد سلع ضارة بالصحة بموجب قانون حماية المستهلك؟",
        "expected": {"source": "lloc", "doc_id": "K3512", "article_no": "19"},
    },
    {  # penalty clause -- Central Bank of Bahrain law
        "question": "ما عقوبة مخالفة احكام السرية المصرفية بموجب قانون مصرف البحرين المركزي؟",
        "expected": {"source": "lloc", "doc_id": "K6406", "article_no": "167"},
    },
    {  # penalty clause -- chemical weapons prohibition law
        "question": "ما عقوبة مخالفة احكام قانون حظر الاسلحة الكيميائية؟",
        "expected": {"source": "lloc", "doc_id": "K5109", "article_no": "19"},
    },
    {  # penalty clause -- protection of society from terrorist acts law
        "question": "ما عقوبة الابلاغ كذبا عن جريمة ارهابية بموجب قانون حماية المجتمع من الاعمال الارهابية؟",
        "expected": {"source": "lloc", "doc_id": "K5806", "article_no": "19"},
    },
    {  # penalty clause -- Penal Code
        "question": "ما عقوبة من يحلف يمينا كاذبة في دعوى مدنية بموجب قانون العقوبات؟",
        "expected": {"source": "lloc", "doc_id": "L1576", "article_no": "239"},
    },
    {  # quorum -- Commercial Companies Law
        "question": "متى يعتبر اجتماع الجمعية العامة العادية لشركة تجارية صحيحا؟",
        "expected": {"source": "lloc", "doc_id": "L2101", "article_no": "243"},
    },
    {  # license non-transfer -- home daycare regulation
        "question": "هل يجوز التنازل عن ترخيص الحضانة المنزلية لشخص اخر؟",
        "expected": {"source": "lloc", "doc_id": "RSOC1006", "article_no": "10"},
    },
    {  # incompatibility -- Judicial Authority Law
        "question": "هل يجوز للقضاة الجمع بين وظيفة القضاء وعمل تجاري؟",
        "expected": {"source": "lloc", "doc_id": "L4202", "article_no": "27"},
    },
    {  # incompatibility -- Shura & Representatives Councils Law
        "question": "هل يجوز الجمع بين عضوية مجلس الشورى وعضوية مجلس النواب؟",
        "expected": {"source": "lloc", "doc_id": "L1502", "article_no": "34"},
    },
    {  # incompatibility -- charity foundation licensing decision
        "question": "هل يجوز الجمع بين عضوية مجلس امناء مؤسسة خيرية ومؤسسة اخرى مماثلة؟",
        "expected": {"source": "lloc", "doc_id": "RLSD9021", "article_no": "١٣"},
    },
    {  # incompatibility -- disability rest-hours regulation
        "question": "هل يجوز الجمع بين ساعتي الراحة المقررة لذوي الاعاقة وساعات الرعاية الاخرى؟",
        "expected": {"source": "lloc", "doc_id": "RLSD8018", "article_no": "10"},
    },
    {  # incompatibility -- youth/sports clubs law
        "question": "هل يجوز الجمع بين عضوية اكثر من ناد او اتحاد رياضي واحد؟",
        "expected": {"source": "lloc", "doc_id": "L5010", "article_no": "60"},
    },
    {  # funds revert to treasury -- marine sand extraction law
        "question": "الى اين تؤول حصيلة بيع الرمال البحرية المستخرجة؟",
        "expected": {"source": "lloc", "doc_id": "K3714", "article_no": "5"},
    },
    {  # appeal restriction -- Cassation Court Law
        "question": "هل يجوز الطعن بطريق التمييز في الاحكام الصادرة قبل الفصل في موضوع الدعوى؟",
        "expected": {"source": "lloc", "doc_id": "L2315", "article_no": "4"},
    },
    # ============================================================================
    # SECOND DOUBLING: 20 more substantive + 26 more boilerplate, same rigor as before
    # (every article_no cross-checked programmatically against the visible number in the
    # text itself, not just eyeballed) -- doubles both categories to see if the 100%
    # vs. 35% split holds up at 2x the sample size, or was itself partly noise.
    # ============================================================================
    # --- 20 more substantive, topic-specific questions ---
    {
        "question": "هل يجوز للملك حل مجلس النواب للسبب ذاته مرتين؟",
        "expected": {"source": "lloc", "doc_id": "ConstAmend2012", "article_no": "42"},
    },
    {
        "question": "ماذا يجب على ادارة مؤسسة الاصلاح والتاهيل فعله عند وفاة نزيل؟",
        "expected": {"source": "lloc", "doc_id": "K1814", "article_no": "7"},
    },
    {
        "question": "هل يجوز صرف نفقة مؤقتة من صندوق النفقة قبل صدور حكم بتقرير النفقة؟",
        "expected": {"source": "lloc", "doc_id": "K3405", "article_no": "5"},
    },
    {
        "question": "هل يجوز التنازل عن الدعوى الجنائية او وقفها في غير الاحوال المبينة قانونا؟",
        "expected": {"source": "lloc", "doc_id": "L4602", "article_no": "7"},
    },
    {
        "question": "ما الشروط الواجب توافرها فيمن يعين عضوا بالمحكمة الدستورية؟",
        "expected": {"source": "lloc", "doc_id": "L2702", "article_no": "4"},
    },
    {
        "question": "ما الشرط الواجب توافره فيمن يمارس المحاماة امام المحاكم؟",
        "expected": {"source": "lloc", "doc_id": "L2680", "article_no": "1"},
    },
    {
        "question": "هل يجوز لكاتب العدل توثيق محرر يخص احد اقاربه الى الدرجة الرابعة؟",
        "expected": {"source": "lloc", "doc_id": "L1471", "article_no": "3"},
    },
    {
        "question": "هل يجوز مزاولة مهنة تدقيق الحسابات دون القيد في السجل؟",
        "expected": {"source": "lloc", "doc_id": "L1521", "article_no": "2"},
    },
    {
        "question": "ما هي نسبة ضريبة القيمة المضافة المفروضة على السلع والخدمات؟",
        "expected": {"source": "lloc", "doc_id": "L4818", "article_no": "3"},
    },
    {
        "question": "هل يجوز اتمام عمليات التركيز الاقتصادي دون موافقة هيئة المنافسة؟",
        "expected": {"source": "lloc", "doc_id": "K3118", "article_no": "12"},
    },
    {
        "question": "ما الشرط الواجب توافره للحصول على ترخيص المستودع الضريبي؟",
        "expected": {"source": "lloc", "doc_id": "K4017", "article_no": "11"},
    },
    {
        "question": "ما هي المدة القصوى لعهدة مالية عادية؟",
        "expected": {"source": "lloc", "doc_id": "L2316", "article_no": "13"},
    },
    {
        "question": "هل يجوز ان يكون للتاجر اكثر من اسم تجاري واحد؟",
        "expected": {"source": "lloc", "doc_id": "K1812", "article_no": "9"},
    },
    {
        "question": "هل يجوز استملاك عقار دون تعويض عادل او لغير المنفعة العامة؟",
        "expected": {"source": "lloc", "doc_id": "K3909", "article_no": "2"},
    },
    {
        "question": "كيف يتم تصنيف السلع والخدمات عند تسجيل العلامات التجارية؟",
        "expected": {"source": "lloc", "doc_id": "K1106", "article_no": "9"},
    },
    {
        "question": "هل يحق لصاحب الاسرار التجارية منع الغير من التعدي عليها؟",
        "expected": {"source": "lloc", "doc_id": "K0703", "article_no": "3"},
    },
    {
        "question": "هل يجوز تغيير الشكل القانوني للمطور العقاري قبل تسليم مشروع التطوير؟",
        "expected": {"source": "lloc", "doc_id": "K2814", "article_no": "9"},
    },
    {
        "question": "هل يجوز جمع المال للاغراض العامة دون ترخيص من الوزير؟",
        "expected": {"source": "lloc", "doc_id": "L2113", "article_no": "2"},
    },
    {
        "question": "هل يلتزم المقيد في السجل التجاري بالحصول على التراخيص اللازمة لمزاولة نشاطه؟",
        "expected": {"source": "lloc", "doc_id": "L2715", "article_no": "8"},
    },
    {
        "question": "هل يجوز الجمع بين ذكر وانثى في غرفة واحدة عند استقدام فنانين اجانب؟",
        "expected": {"source": "lloc", "doc_id": "RINF0391", "article_no": "5"},
    },
    # --- 26 more boilerplate/generic-clause questions ---
    {  # effective date
        "question": "متى يعمل بقانون الصناعات والمهن الخطرة والمضرة بالصحة؟",
        "expected": {"source": "lloc", "doc_id": "RHEL0577", "article_no": "3"},
    },
    {  # penalty -- juveniles law
        "question": "ما عقوبة اخفاء حدث حكم بتسليمه لشخص او جهة او مساعدته على الفرار؟",
        "expected": {"source": "lloc", "doc_id": "L1776", "article_no": "21"},
    },
    {  # penalty -- securities brokerage
        "question": "ما عقوبة مزاولة مهنة دلالة الاوراق المالية بدون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "L0682", "article_no": "8"},
    },
    {  # penalty -- narcotics
        "question": "ما عقوبة تعاطي المؤثرات العقلية في غير الاحوال المرخص بها بموجب قانون المواد المخدرة؟",
        "expected": {"source": "lloc", "doc_id": "K1507", "article_no": "35"},
    },
    {  # penalty -- political societies
        "question": "ما عقوبة تسلم جمعية سياسية اموالا من جهة غير بحرينية لحسابها؟",
        "expected": {"source": "lloc", "doc_id": "K2605", "article_no": "24"},
    },
    {  # penalty -- assisted reproduction
        "question": "ما عقوبة مخالفة احكام قانون التقنيات الطبية المساعدة على التلقيح الاصطناعي؟",
        "expected": {"source": "lloc", "doc_id": "K2617", "article_no": "16"},
    },
    {  # penalty -- HIV protection law
        "question": "ما عقوبة التمييز ضد المتعايشين مع فيروس نقص المناعة المكتسب؟",
        "expected": {"source": "lloc", "doc_id": "K0117", "article_no": "23"},
    },
    {  # penalty -- domestic violence protection
        "question": "ما عقوبة مخالفة امر الحماية الصادر بموجب قانون الحماية من العنف الاسري؟",
        "expected": {"source": "lloc", "doc_id": "K1715", "article_no": "16"},
    },
    {  # penalty -- the OLD, since-superseded 1976 Labor Law (project-relevant: this is exactly
       # the kind of repealed-article-reuse risk flagged elsewhere in this project)
        "question": "ما عقوبة صاحب العمل الذي يعترض على قيام موظفي التفتيش بمهامهم بموجب قانون العمل الصادر عام 1976؟",
        "expected": {"source": "lloc", "doc_id": "L2376", "article_no": "168"},
    },
    {  # penalty -- child law, online exploitation
        "question": "ما عقوبة استدراج واستغلال الاطفال عبر الانترنت بموجب قانون الطفل؟",
        "expected": {"source": "lloc", "doc_id": "K3712", "article_no": "66"},
    },
    {  # penalty -- health precautions law
        "question": "ما عقوبة مخالفة قانون الاحتياطات الصحية للوقاية من الامراض المعدية؟",
        "expected": {"source": "lloc", "doc_id": "L1477", "article_no": "13"},
    },
    {  # quorum -- housing bank
        "question": "متى يعتبر اجتماع مجلس ادارة بنك الاسكان صحيحا؟",
        "expected": {"source": "lloc", "doc_id": "L0479", "article_no": "13"},
    },
    {  # incompatibility -- the Constitution itself
        "question": "هل يجوز الجمع بين عضوية مجلس الشورى ومجلس النواب بموجب الدستور؟",
        "expected": {"source": "lloc", "doc_id": "Constitution", "article_no": "97"},
    },
    {  # incompatibility -- charitable foundation board
        "question": "هل يجوز الجمع بين عضوية مجلس امناء مؤسسة والعمل فيها باجر؟",
        "expected": {"source": "lloc", "doc_id": "RSOCD1015", "article_no": "13"},
    },
    {  # incompatibility -- clubs/associations law
        "question": "هل يجوز الجمع بين عضوية مجلس ادارة جمعيتين تعملان في ميدان واحد؟",
        "expected": {"source": "lloc", "doc_id": "L2189", "article_no": "42"},
    },
    {  # license non-transfer -- pharmacy law
        "question": "هل يجوز التنازل عن ترخيص فتح مركز صيدلي للغير؟",
        "expected": {"source": "lloc", "doc_id": "L1897", "article_no": "15"},
    },
    {  # withdrawal -- civil procedures law
        "question": "هل يجوز للخصم سحب مستند قدمه للاستدلال به في الدعوى دون رضا خصمه؟",
        "expected": {"source": "lloc", "doc_id": "L1271", "article_no": "146"},
    },
    {  # withdrawal -- disability rights convention reservations
        "question": "هل يجوز سحب التحفظات على اتفاقية حقوق الاشخاص ذوي الاعاقة في اي وقت؟",
        "expected": {"source": "lloc", "doc_id": "K2211", "article_no": "46"},
    },
    {  # withdrawal -- commercial law, bills of exchange
        "question": "هل يجوز سحب الكمبيالة لحساب شخص اخر؟",
        "expected": {"source": "lloc", "doc_id": "L0787", "article_no": "352"},
    },
    {  # withdrawal -- GCC patent system
        "question": "هل يجوز لمقدم طلب براءة الاختراع سحب طلبه قبل البت فيه بصفة نهائية؟",
        "expected": {"source": "lloc", "doc_id": "K1204", "article_no": "8"},
    },
    {  # withdrawal -- unified GCC customs law
        "question": "هل يجوز اتخاذ تدابير لسحب البضائع عند اعلان حالة الطوارئ؟",
        "expected": {"source": "lloc", "doc_id": "L1002", "article_no": "65"},
    },
    {  # appeal deadline -- maritime law
        "question": "خلال كم يوما يجوز الطعن في حكم رسو المزاد بموجب القانون البحري؟",
        "expected": {"source": "lloc", "doc_id": "K1022", "article_no": "67"},
    },
    {  # appeal deadline -- municipal fees regulation
        "question": "خلال كم يوما يجوز التظلم من الرسوم البلدية؟",
        "expected": {"source": "lloc", "doc_id": "RCAB1602", "article_no": "63"},
    },
    {  # appeal deadline -- groundwater regulation
        "question": "خلال كم يوما يجب تقديم التظلم من قرار مكتب مصادر المياه؟",
        "expected": {"source": "lloc", "doc_id": "L1280", "article_no": "17"},
    },
    {  # appeal deadline -- industry regulation law
        "question": "خلال كم يوما يجوز الطعن امام المحكمة المدنية الكبرى في قرار رفض التظلم بشأن تنظيم الصناعة؟",
        "expected": {"source": "lloc", "doc_id": "L0684", "article_no": "24"},
    },
    {  # board formation -- water resources council
        "question": "كيف يشكل مجلس الموارد المائية؟",
        "expected": {"source": "lloc", "doc_id": "L0782", "article_no": "2"},
    },
]
len(EVAL_SET)

## Retrieval evaluation

Each question here has exactly **one** correct source, not a set of several — so precision@k isn't the right metric (it would just measure how much of `k` is "wasted" on other passages, which we don't actually care about for a single-answer lookup). The right metrics for this shape of eval are:

- **Hit@k** — did the correct passage appear anywhere in the top `k` results?
- **MRR (Mean Reciprocal Rank)** — *where* did it rank when found? 1.0 if it was the very top result, 0.5 if second, 0 if not found at all within `k`.

**Important:** `max_marginal_relevance_search`'s default `fetch_k` is only 20 — but the real deployed app (`app.py`'s `ThresholdMMRRetriever`) always uses `fetch_k=300`, calibrated earlier in this project specifically because Chroma's approximate search was found to miss the correct result entirely at low candidate-pool sizes. Testing with the library default instead of the production value would measure a weaker configuration than what's actually deployed — so this eval explicitly matches production's `fetch_k=300`.

In [ ]:
SEARCH_FETCH_POOL = 300  # matches app.py's calibrated production value, not the library default of 20

def matches(doc, expected):
    m = doc.metadata
    return (
        m.get("source") == expected["source"]
        and m.get("doc_id") == expected["doc_id"]
        and str(m.get("article_no")) == str(expected.get("article_no"))
    )


def evaluate_retrieval_one(question, expected, k=6, fetch_k=SEARCH_FETCH_POOL):
    docs = vectordb.max_marginal_relevance_search(question, k=k, fetch_k=fetch_k)
    for rank, doc in enumerate(docs, start=1):
        if matches(doc, expected):
            return {"found": True, "rank": rank, "reciprocal_rank": 1 / rank}
    return {"found": False, "rank": None, "reciprocal_rank": 0.0}

In [ ]:
import pandas as pd

retrieval_rows = []
for item in EVAL_SET:
    result = evaluate_retrieval_one(item["question"], item["expected"], k=6)
    retrieval_rows.append({
        "question": item["question"],
        "expected_doc_id": item["expected"]["doc_id"],
        "expected_article": item["expected"]["article_no"],
        **result,
    })

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df

In [ ]:
hit_rate_at_6 = retrieval_df["found"].mean()
mrr = retrieval_df["reciprocal_rank"].mean()

print(f"Hit@6:  {hit_rate_at_6:.1%}  ({retrieval_df['found'].sum()}/{len(retrieval_df)} questions)")
print(f"MRR:    {mrr:.3f}")

## Generation evaluation

Retrieval finding the right passage doesn't guarantee the generated answer actually cites it correctly — today's Groq test showed a model can retrieve the right text and still wrap it in a fabricated citation. This section checks, for each question: does the generated answer's text contain the expected article number? This is a **necessary-but-not-sufficient** check — it catches a missing citation, but a model could still name the *right* article number while attributing it to the *wrong* law or case (exactly what happened in the Groq test). Catching that reliably needs either a stricter structured-citation check or a human/SME spot check — noted in "How to extend this" below.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).
- اذا استندت الاجابة الى اكثر من قانون او حكم، اذكرهم جميعا.

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 6}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)

In [ ]:
def evaluate_generation_one(question, expected):
    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    article_cited = str(expected["article_no"]) in answer
    doc_id_cited = expected["doc_id"] in answer
    return {
        "answer": answer,
        "article_number_present": article_cited,
        "doc_id_present": doc_id_cited,
        "citation_present": article_cited or doc_id_cited,
    }

In [ ]:
generation_rows = []
for item in EVAL_SET:
    result = evaluate_generation_one(item["question"], item["expected"])
    generation_rows.append({
        "question": item["question"],
        "expected_doc_id": item["expected"]["doc_id"],
        "expected_article": item["expected"]["article_no"],
        **result,
    })

generation_df = pd.DataFrame(generation_rows)
generation_df[["question", "expected_doc_id", "expected_article", "citation_present"]]

In [ ]:
citation_accuracy = generation_df["citation_present"].mean()
print(f"Citation present: {citation_accuracy:.1%}  ({generation_df['citation_present'].sum()}/{len(generation_df)} questions)")

### Read a full answer

Worth reading at least one full generated answer alongside its expected citation, not just the pass/fail flag — the flag only proves a number appeared somewhere in the text, not that it was used correctly.

In [ ]:
i = 0
print("QUESTION:", generation_df.loc[i, "question"])
print("EXPECTED:", generation_df.loc[i, "expected_doc_id"], generation_df.loc[i, "expected_article"])
print()
print(generation_df.loc[i, "answer"])

## Summary

In [ ]:
summary = pd.DataFrame([
    {"metric": "Retrieval Hit@6", "value": f"{hit_rate_at_6:.1%}"},
    {"metric": "Retrieval MRR", "value": f"{mrr:.3f}"},
    {"metric": "Generation citation-present rate", "value": f"{citation_accuracy:.1%}"},
])
summary

## How to extend this

- **Grow the eval set further**, especially with real client-provided questions once available (the original open question to the client that this was always blocked on) — 26 questions is enough to catch a broken pipeline and spot a field-specific weak point, not enough to trust a percentage as a stable, statistically confident number.
- **Add `sjc` case-law questions**, not just `lloc` legislation — this set is legislation-only because those citations were the fastest to verify directly against the corpus; case citations need the same direct-lookup discipline before being trusted as ground truth.
- **Tighten the generation check**: right now it only checks whether the expected article number appears anywhere in the text, which would not have caught today's Groq bug (the correct article number appeared, wrapped in a fabricated law name). A stricter check would parse out the model's actual cited law/case name and compare it against the expected `doc_id`'s real title, not just search for a number.
- **LLM-as-judge or a human/SME spot check** for anything number-matching can't catch — no automated check here substitutes for a real lawyer reading a sample of answers.
- **LangSmith**, if useful later: this same logic (eval set + scoring functions) can be uploaded as a LangSmith dataset with these functions as evaluators, to get a dashboard and run-over-run comparison instead of a one-off notebook run — optional tooling on top of the same metrics, not a replacement for them.